In [1]:
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent / "src"

print("Looking in:", BASE_DIR)

df_l2 = pd.read_excel(BASE_DIR / "L2ReadingData.xlsx")
print(df_l2.head())

Looking in: /Users/emmamaenhout/Desktop/Channel Islands/Spring 2026/Machine Learning/mlclass/src
  PP_NR      GROUP LANGUAGE_RANK LANGUAGE  PART  TRIAL  TRIAL_FIXATION_COUNT  \
0  pp01  bilingual            L2  English     3      5                   125   
1  pp01  bilingual            L2  English     3      5                   125   
2  pp01  bilingual            L2  English     3      5                   125   
3  pp01  bilingual            L2  English     3      5                   125   
4  pp01  bilingual            L2  English     3      5                   125   

   TRIAL_TOTAL_READING_TIME  WORD_ID_WITHIN_TRIAL WORD_ID  ...  \
0                     27561                     1   3-5-1  ...   
1                     27561                     2   3-5-2  ...   
2                     27561                     3   3-5-3  ...   
3                     27561                     4   3-5-4  ...   
4                     27561                     5   3-5-5  ...   

  WORD_LAST_FIXATION_RUN 

In [6]:
df_l2['WORD_AVERAGE_FIX_PUPIL_SIZE'] = pd.to_numeric(
    df_l2['WORD_AVERAGE_FIX_PUPIL_SIZE'],
    errors='coerce'
)

In [7]:
df_l2['WORD_FIXATION_%'] = (
    df_l2['WORD_FIXATION_%']
    .astype(str)
    .str.replace('%', '', regex=False)
)

df_l2['WORD_FIXATION_%'] = pd.to_numeric(
    df_l2['WORD_FIXATION_%'],
    errors='coerce'
)

In [8]:
cols_to_numeric = [
    'WORD_GAZE_DURATION',
    'WORD_AVERAGE_FIX_PUPIL_SIZE',
    'WORD_SKIP',
    'WORD_FIXATION_%',
    'WORD_TOTAL_READING_TIME'
]

for col in cols_to_numeric:
    df_l2[col] = pd.to_numeric(df_l2[col], errors='coerce')

In [9]:
trial_level = df_l2.groupby(
    ['PP_NR', 'TRIAL']
).agg({
    'WORD_GAZE_DURATION': ['mean', 'std', 'max'],
    'WORD_AVERAGE_FIX_PUPIL_SIZE': ['mean', 'std'],
    'WORD_SKIP': 'mean',
    'WORD_FIXATION_%': 'mean',
    'WORD_TOTAL_READING_TIME': 'sum'
}).reset_index()


In [10]:
trial_level.columns = [
    '_'.join(col).strip('_') for col in trial_level.columns
]

In [11]:
print(trial_level)

     PP_NR  TRIAL  WORD_GAZE_DURATION_mean  WORD_GAZE_DURATION_std  \
0     pp01      5               283.583333              166.311870   
1     pp01      6               255.069307              119.273489   
2     pp01      7               256.327381              118.122564   
3     pp01      8               267.442857              120.501725   
4     pp01      9               252.522293              119.468781   
...    ...    ...                      ...                     ...   
2835  pp19    141               224.663551              135.611400   
2836  pp19    142               201.842105               80.579772   
2837  pp19    143               201.040816               76.334232   
2838  pp19    144               220.053333               91.171785   
2839  pp19    145               210.785714               89.277039   

      WORD_GAZE_DURATION_max  WORD_AVERAGE_FIX_PUPIL_SIZE_mean  \
0                      989.0                       2580.544295   
1                      605.

In [12]:
trial_level = trial_level.fillna(0)

In [13]:
X = trial_level.drop(columns=['PP_NR', 'TRIAL'])

In [14]:
trial_level['breakdown'] = (
    trial_level['WORD_TOTAL_READING_TIME_sum'] >
    trial_level['WORD_TOTAL_READING_TIME_sum'].quantile(0.85)
).astype(int)

In [15]:
y = trial_level['breakdown']

In [16]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [19]:
class BreakdownMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.model(x)

In [21]:
plt.figure()
plt.plot(hist["train_loss"], label="train loss")
plt.plot(hist["test_loss"], label="test loss")

NameError: name 'hist' is not defined

<Figure size 640x480 with 0 Axes>